# Email Spam/Ham Classifier

## 1. Import Libraries

First, we'll import all the necessary libraries for data manipulation, machine learning, and evaluation.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 2. Data Collection & Pre-Processing

We load the email dataset, handle any potential null values, and convert the 'spam'/'ham' categories into numerical labels (0 and 1) for the machine learning model.

In [2]:
# Loading the data from csv file to a pandas Dataframe
raw_mail_data = pd.read_csv('mail_data.csv')

# Replace the null values with a null string
mail_data = raw_mail_data.where((pd.notnull(raw_mail_data)), '')

# Label Encoding: Label spam mail as 0; Non-spam mail (ham) as 1
mail_data.loc[mail_data['Category'] == 'spam', 'Category'] = 0
mail_data.loc[mail_data['Category'] == 'ham', 'Category'] = 1

# Separating the data as text and labels
X = mail_data['Message']
Y = mail_data['Category']

display(mail_data.head())

,Category,Message
0,1,"Go until jurong point, crazy.. Available only ..."
1,1,Ok lar... Joking wif u oni...
2,0,Free entry in 2 a wkly comp to win FA Cup fina...
3,1,U dun say so early hor... U c already then say...
4,1,"Nah I don't think he goes to usf, he lives aro..."


## 3. Splitting the Data into Training Data and Test Data

We split the dataset into training and testing sets to evaluate the model's performance on unseen data. 80% of the data will be used for training and 20% for testing.

In [3]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=3)

## 4. Feature Extraction

Since machine learning models cannot directly process text, we use `TfidfVectorizer` to convert the email messages into numerical feature vectors. This process also removes common English stop words and converts text to lowercase.

In [4]:
# Transform the text data to feature vectors that can be used as input to the Logistic regression model
feature_extraction = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)

X_train_features = feature_extraction.fit_transform(X_train)
X_test_features = feature_extraction.transform(X_test)

# Convert Y_train and Y_test values as integers
Y_train = Y_train.astype('int')
Y_test = Y_test.astype('int')

## 5. Training the Logistic Regression Model

We will use a Logistic Regression model, a common choice for binary classification tasks like spam detection, and train it with our extracted features.

In [5]:
model = LogisticRegression()

# Training the Logistic Regression model with the training data
model.fit(X_train_features, Y_train)

LogisticRegression()

## 6. Evaluating the Trained Model

We assess the model's performance by calculating its accuracy on both the training data (to check for overfitting) and the test data (to gauge generalization).

In [6]:
# Prediction on training data
prediction_on_training_data = model.predict(X_train_features)
accuracy_on_training_data = accuracy_score(Y_train, prediction_on_training_data)
print('Accuracy on training data : ', accuracy_on_training_data)

# Prediction on test data
prediction_on_test_data = model.predict(X_test_features)
accuracy_on_test_data = accuracy_score(Y_test, prediction_on_test_data)
print('Accuracy on test data : ', accuracy_on_test_data)

Accuracy on training data :  0.9676912721561588
Accuracy on test data :  0.9668161434977578


## 7. Save the Trained Model and Feature Extractor

To avoid retraining the model every time, we'll save both the `LogisticRegression` model and the `TfidfVectorizer` (feature extractor) using Python's `pickle` library. This allows us to load them directly for future predictions.

In [8]:
import pickle

# Save the trained Logistic Regression model
with open('spam_ham_model.pkl', 'wb') as file:
    pickle.dump(model, file)
print('Trained model saved as spam_ham_model.pkl')

# Save the TfidfVectorizer (feature extractor)
with open('tfidf_vectorizer.pkl', 'wb') as file:
    pickle.dump(feature_extraction, file)
print('TfidfVectorizer saved as tfidf_vectorizer.pkl')

Trained model saved as spam_ham_model.pkl
TfidfVectorizer saved as tfidf_vectorizer.pkl


## 8. Building a Predictive System

Finally, we demonstrate how to use the trained model to predict whether a new, unseen email is 'spam' or 'ham'.

**Note:** To use the saved model and vectorizer in a new session, you would load them like this:
```python
# import pickle
# with open('spam_ham_model.pkl', 'rb') as file:
#     loaded_model = pickle.load(file)
# with open('tfidf_vectorizer.pkl', 'rb') as file:
#     loaded_feature_extraction = pickle.load(file)
```

## 7. Building a Predictive System

Finally, we demonstrate how to use the trained model to predict whether a new, unseen email is 'spam' or 'ham'.

In [7]:
# Add any text inside the quotes to test the system
input_mail = ["I've been searching for the right words to thank you for this breather. I promise i wont take your help for granted and will fulfil my promise. You have been wonderful and a blessing at all times."]

# Convert text to feature vectors
input_data_features = feature_extraction.transform(input_mail)

# Making prediction
prediction = model.predict(input_data_features)

# Output prediction result
if (prediction[0] == 1):
  print('Ham mail')
else:
  print('Spam mail')

Ham mail
